In [3]:
import pandas as pd
from typing import Tuple
import math
import numpy as np
from tqdm import tqdm
from collections import deque, Counter
from pathlib import Path

# Cleaning Game Info
This notebook provides a clean version of the raw `data/raw/gameinfo.csv` file.
If does the following:
1. Filtering for competitive games.
2. Adding a 'homewon' column, that is True if the home team won and False otherwise.
3. Adding a 'hometeamgamecount' and 'visteamgamecount' column, which gives a count of how many games have been played all time for the home and visiting teams, including the current game.
4. Adding a 'marginofvictory' column.
5. Adding a 'timestamp' column, which provides a `pd.Timestamp` for the game start time.
6. Adding 'homedistancetraveled' and 'visdistancetraveled' columns, which contains the distance in miles traveled from the team's previous games.
7. Adding 'homerestdays' and 'visrestdays' columns, which contains the number of days between the current game and the previous game for the home and visiting teams.
8. Adding 'homepitcherrgs', 'vispitcherrgs', 'hometeamrgs', 'visteamrgs', 'homepitcherminusteamrgs', 'vispitcherminusteamrgs', 'homefirstpitchergameofseason', and 'visfirstpitchergameofseason' columns, which contain running averages of game scores for the individual pitcher up to but not including the game as well as for the overall team, and their differences, and whether that was
the pitcher's first game of the season.
9.  Add 'homewinstreak' and 'viswinstreak' columns, containing the win streak of the home and away teams.
10. Save to a new csv, `data/cleangameinfo_clean.csv`

In [2]:
pd.set_option('display.max_columns', None)

In [11]:
all_games = pd.read_csv(Path.cwd() / "analysis" / "data" / "raw" / "gameinfo.csv")
all_games.head()

/var/folders/f3/w3vlb72x47z3vhjsm9s6jlxc0000gn/T/ipykernel_26154/2861852725.py:1: DtypeWarning: Columns (10,11,13,17,19,21,27,28) have mixed types. Specify dtype option on import or set low_memory=False.
  all_games = pd.read_csv(Path.cwd() / "analysis" / "data" / "raw" / "gameinfo.csv")


,gid,visteam,hometeam,site,date,number,starttime,daynight,innings,tiebreaker,usedh,htbf,timeofgame,attendance,fieldcond,precip,sky,temp,winddir,windspeed,oscorer,forfeit,suspend,umphome,ump1b,ump2b,ump3b,umplf,umprf,wp,lp,save,gametype,vruns,hruns,wteam,lteam,line,batteries,lineups,box,pbp,season
0,CIN189804150,CL4,CIN,CIN05,18980415,0.0,0:00PM,day,NaN,NaN,False,True,100.0,11000.0,unknown,unknown,unknown,0.0,unknown,-1.0,NaN,NaN,NaN,sware101,woodg104,NaN,NaN,NaN,NaN,breit101,younc102,NaN,regular,2,3,CIN,CL4,y,both,y,y,NaN,1898
1,LS3189804150,PIT,LS3,LOU03,18980415,0.0,0:00PM,day,NaN,NaN,False,True,80.0,10000.0,unknown,unknown,unknown,0.0,unknown,-1.0,NaN,NaN,NaN,cushc801,heydj901,NaN,NaN,NaN,NaN,cunnb103,killf101,NaN,regular,3,10,LS3,PIT,y,both,y,y,NaN,1898
2,SLN189804150,CHN,SLN,STL05,18980415,0.0,0:00PM,day,NaN,NaN,False,NaN,105.0,12000.0,unknown,unknown,unknown,0.0,unknown,-1.0,NaN,NaN,NaN,mcdoj105,odayh101,NaN,NaN,NaN,NaN,grifc101,taylj102,NaN,regular,2,1,CHN,SLN,y,both,y,y,NaN,1898
3,BLN189804160,WSN,BLN,BAL07,18980416,0.0,0:00PM,day,NaN,NaN,False,NaN,135.0,6518.0,unknown,unknown,unknown,0.0,unknown,-1.0,NaN,NaN,NaN,lynct901,connt901,NaN,NaN,NaN,NaN,mcjad101,weyhg101,NaN,regular,3,8,BLN,WSN,y,both,y,y,NaN,1898
4,CIN189804160,CL4,CIN,CIN05,18980416,0.0,0:00PM,day,NaN,NaN,False,True,110.0,6504.0,unknown,unknown,unknown,0.0,unknown,-1.0,NaN,NaN,NaN,woodg104,sware101,NaN,NaN,NaN,NaN,powej105,hillb102,NaN,regular,3,1,CL4,CIN,y,both,y,y,NaN,1898


## 1. Filter for Competitive Games

In [82]:
valid_games = ['regular', 'championship', 'worldseries', 'lcs',
                'playoff', 'divisionseries', 'wildcard']

all_games = all_games[all_games['gametype'].isin(valid_games)].reset_index(drop=True)

## 2. Add 'homewon' Column

In [83]:
all_games['homewon'] = list((all_games['hruns'] > all_games['vruns']))
all_games.head()

,gid,visteam,hometeam,site,date,number,starttime,daynight,innings,tiebreaker,usedh,htbf,timeofgame,attendance,fieldcond,precip,sky,temp,winddir,windspeed,oscorer,forfeit,suspend,umphome,ump1b,ump2b,ump3b,umplf,umprf,wp,lp,save,gametype,vruns,hruns,wteam,lteam,line,batteries,lineups,box,pbp,season,homewon
0,CIN189804150,CL4,CIN,CIN05,18980415,0.0,0:00PM,day,NaN,NaN,False,True,100.0,11000.0,unknown,unknown,unknown,0.0,unknown,-1.0,NaN,NaN,NaN,sware101,woodg104,NaN,NaN,NaN,NaN,breit101,younc102,NaN,regular,2,3,CIN,CL4,y,both,y,y,NaN,1898,True
1,LS3189804150,PIT,LS3,LOU03,18980415,0.0,0:00PM,day,NaN,NaN,False,True,80.0,10000.0,unknown,unknown,unknown,0.0,unknown,-1.0,NaN,NaN,NaN,cushc801,heydj901,NaN,NaN,NaN,NaN,cunnb103,killf101,NaN,regular,3,10,LS3,PIT,y,both,y,y,NaN,1898,True
2,SLN189804150,CHN,SLN,STL05,18980415,0.0,0:00PM,day,NaN,NaN,False,NaN,105.0,12000.0,unknown,unknown,unknown,0.0,unknown,-1.0,NaN,NaN,NaN,mcdoj105,odayh101,NaN,NaN,NaN,NaN,grifc101,taylj102,NaN,regular,2,1,CHN,SLN,y,both,y,y,NaN,1898,False
3,BLN189804160,WSN,BLN,BAL07,18980416,0.0,0:00PM,day,NaN,NaN,False,NaN,135.0,6518.0,unknown,unknown,unknown,0.0,unknown,-1.0,NaN,NaN,NaN,lynct901,connt901,NaN,NaN,NaN,NaN,mcjad101,weyhg101,NaN,regular,3,8,BLN,WSN,y,both,y,y,NaN,1898,True
4,CIN189804160,CL4,CIN,CIN05,18980416,0.0,0:00PM,day,NaN,NaN,False,True,110.0,6504.0,unknown,unknown,unknown,0.0,unknown,-1.0,NaN,NaN,NaN,woodg104,sware101,NaN,NaN,NaN,NaN,powej105,hillb102,NaN,regular,3,1,CL4,CIN,y,both,y,y,NaN,1898,False


## 3. Add 'hometeamgamecount' and 'visteamgamecount' columns

In [84]:
count = Counter()

hometeamgamecounts = []
visteamgamecounts = []

for _, game in all_games.iterrows():
    home_team = game['hometeam']
    away_team = game['visteam']

    count[home_team] += 1
    count[away_team] += 1

    home_team_count = count[home_team]
    away_team_count = count[away_team]

    hometeamgamecounts.append(home_team_count)
    visteamgamecounts.append(away_team_count)

all_games['hometeamgamecount'] = hometeamgamecounts
all_games['visteamgamecount'] = visteamgamecounts

## 4. Add 'marginofvictory' column

In [85]:
all_games['marginofvictory'] = abs((all_games['hruns'] - all_games['vruns']))
all_games.head()

,gid,visteam,hometeam,site,date,number,starttime,daynight,innings,tiebreaker,usedh,htbf,timeofgame,attendance,fieldcond,precip,sky,temp,winddir,windspeed,oscorer,forfeit,suspend,umphome,ump1b,ump2b,ump3b,umplf,umprf,wp,lp,save,gametype,vruns,hruns,wteam,lteam,line,batteries,lineups,box,pbp,season,homewon,hometeamgamecount,visteamgamecount,marginofvictory
0,CIN189804150,CL4,CIN,CIN05,18980415,0.0,0:00PM,day,NaN,NaN,False,True,100.0,11000.0,unknown,unknown,unknown,0.0,unknown,-1.0,NaN,NaN,NaN,sware101,woodg104,NaN,NaN,NaN,NaN,breit101,younc102,NaN,regular,2,3,CIN,CL4,y,both,y,y,NaN,1898,True,1,1,1
1,LS3189804150,PIT,LS3,LOU03,18980415,0.0,0:00PM,day,NaN,NaN,False,True,80.0,10000.0,unknown,unknown,unknown,0.0,unknown,-1.0,NaN,NaN,NaN,cushc801,heydj901,NaN,NaN,NaN,NaN,cunnb103,killf101,NaN,regular,3,10,LS3,PIT,y,both,y,y,NaN,1898,True,1,1,7
2,SLN189804150,CHN,SLN,STL05,18980415,0.0,0:00PM,day,NaN,NaN,False,NaN,105.0,12000.0,unknown,unknown,unknown,0.0,unknown,-1.0,NaN,NaN,NaN,mcdoj105,odayh101,NaN,NaN,NaN,NaN,grifc101,taylj102,NaN,regular,2,1,CHN,SLN,y,both,y,y,NaN,1898,False,1,1,1
3,BLN189804160,WSN,BLN,BAL07,18980416,0.0,0:00PM,day,NaN,NaN,False,NaN,135.0,6518.0,unknown,unknown,unknown,0.0,unknown,-1.0,NaN,NaN,NaN,lynct901,connt901,NaN,NaN,NaN,NaN,mcjad101,weyhg101,NaN,regular,3,8,BLN,WSN,y,both,y,y,NaN,1898,True,1,1,5
4,CIN189804160,CL4,CIN,CIN05,18980416,0.0,0:00PM,day,NaN,NaN,False,True,110.0,6504.0,unknown,unknown,unknown,0.0,unknown,-1.0,NaN,NaN,NaN,woodg104,sware101,NaN,NaN,NaN,NaN,powej105,hillb102,NaN,regular,3,1,CL4,CIN,y,both,y,y,NaN,1898,False,2,2,2


## 5. Add 'timestamp' Column

In [86]:
def get_hms(raw_starttime: str) -> Tuple[int, int]:
    """For the given raw start time (e.g. '5:30PM'), returns the hour and minute.
    
    If raw_starttime is nan, this will return 0, 0.
    
    If there is no 'AM' or 'PM' (e.g., '' or '?M'), it will assume a 24 hour clock.
    """

    if pd.isna(raw_starttime):
        return 0, 0
    
    col_idx = raw_starttime.find(':')
    h = int(raw_starttime[:col_idx]) 
    m = int(raw_starttime[col_idx + 1:col_idx + 3])
    
    # If AM or PM, need to do some more conversions
        
    # If time is PM and not 12:00, add 12 hours
    # Note some entries in all_games are 0:00 PM - this would also correctly add 12 hours,
    # making it the familiar 12:00 PM
    if raw_starttime[-2:].upper() == 'PM' and h != 12:
        h += 12
            
    elif raw_starttime[-2:].upper() == 'AM' and h == 12: # Edge case - if midnight, h should be 0
        h = 0
        
    return h, m

def get_timestamp(game: pd.DataFrame) -> pd.Timestamp:
    """For a single row, gets its game start timestamp."""
    #print(game['starttime'])
    h, min = get_hms(game['starttime'])
    
    raw_date = str(game['date'])

    y = int(raw_date[:4])
    mon = int(raw_date[4:6])
    d = int(raw_date[6:])
    return pd.Timestamp(year=y, month=mon, day=d, hour=h, minute=min)

In [87]:
# Add the timestamp column

all_games['timestamp'] = all_games.apply(get_timestamp, axis=1)
all_games.head()

,gid,visteam,hometeam,site,date,number,starttime,daynight,innings,tiebreaker,usedh,htbf,timeofgame,attendance,fieldcond,precip,sky,temp,winddir,windspeed,oscorer,forfeit,suspend,umphome,ump1b,ump2b,ump3b,umplf,umprf,wp,lp,save,gametype,vruns,hruns,wteam,lteam,line,batteries,lineups,box,pbp,season,homewon,hometeamgamecount,visteamgamecount,marginofvictory,timestamp
0,CIN189804150,CL4,CIN,CIN05,18980415,0.0,0:00PM,day,NaN,NaN,False,True,100.0,11000.0,unknown,unknown,unknown,0.0,unknown,-1.0,NaN,NaN,NaN,sware101,woodg104,NaN,NaN,NaN,NaN,breit101,younc102,NaN,regular,2,3,CIN,CL4,y,both,y,y,NaN,1898,True,1,1,1,1898-04-15 12:00:00
1,LS3189804150,PIT,LS3,LOU03,18980415,0.0,0:00PM,day,NaN,NaN,False,True,80.0,10000.0,unknown,unknown,unknown,0.0,unknown,-1.0,NaN,NaN,NaN,cushc801,heydj901,NaN,NaN,NaN,NaN,cunnb103,killf101,NaN,regular,3,10,LS3,PIT,y,both,y,y,NaN,1898,True,1,1,7,1898-04-15 12:00:00
2,SLN189804150,CHN,SLN,STL05,18980415,0.0,0:00PM,day,NaN,NaN,False,NaN,105.0,12000.0,unknown,unknown,unknown,0.0,unknown,-1.0,NaN,NaN,NaN,mcdoj105,odayh101,NaN,NaN,NaN,NaN,grifc101,taylj102,NaN,regular,2,1,CHN,SLN,y,both,y,y,NaN,1898,False,1,1,1,1898-04-15 12:00:00
3,BLN189804160,WSN,BLN,BAL07,18980416,0.0,0:00PM,day,NaN,NaN,False,NaN,135.0,6518.0,unknown,unknown,unknown,0.0,unknown,-1.0,NaN,NaN,NaN,lynct901,connt901,NaN,NaN,NaN,NaN,mcjad101,weyhg101,NaN,regular,3,8,BLN,WSN,y,both,y,y,NaN,1898,True,1,1,5,1898-04-16 12:00:00
4,CIN189804160,CL4,CIN,CIN05,18980416,0.0,0:00PM,day,NaN,NaN,False,True,110.0,6504.0,unknown,unknown,unknown,0.0,unknown,-1.0,NaN,NaN,NaN,woodg104,sware101,NaN,NaN,NaN,NaN,powej105,hillb102,NaN,regular,3,1,CL4,CIN,y,both,y,y,NaN,1898,False,2,2,2,1898-04-16 12:00:00


## 6. Add 'homedistancetraveled' and 'visdistancetraveled' Column
Distance traveled from previous game.

In [88]:
# Add temporary latitude and longitude columns
parks = pd.read_csv('Parks.csv')
all_games = pd.merge(all_games, parks[['PARKID', 'Latitude', 'Longitude']], how='left', left_on='site', right_on='PARKID').reset_index(drop=True)
all_games = all_games.drop('PARKID', axis=1)

In [89]:
def haversine(lat1, lon1, lat2, lon2):
    """Returns Haversine distance between two pairs of latitudes and longitudes."""
    R = 3958.8  # Earth radius in miles
    phi1, phi2 = math.radians(lat1), math.radians(lat2)
    dphi = math.radians(lat2 - lat1)
    dlambda = math.radians(lon2 - lon1)
    a = math.sin(dphi / 2)**2 + math.cos(phi1) * math.cos(phi2) * math.sin(dlambda / 2)**2
    return 2 * R * math.atan2(math.sqrt(a), math.sqrt(1 - a))

In [90]:
def get_closest_home_game_coords(team: str, idx: int) -> Tuple[float, float]:
    """For the given team, returns the latitude and longitude of the closest home game they played
    to wherever they are in all_games, given by idx. If no home game is found, it returns None, None."""
    
    n_games = len(all_games)
    
    before = idx - 1
    after = idx + 1
    while before >= 0 or after <= n_games - 1:
        if before >= 0:
            before_gm = all_games.iloc[before]
            if before_gm['hometeam'] == team:
                return before_gm['Latitude'], before_gm['Longitude']
            
            before -= 1

        if after <= n_games - 1:
            after_gm = all_games.iloc[after]
            if after_gm['hometeam'] == team:
                return after_gm['Latitude'], after_gm['Longitude']
            
            after += 1
            
    return None, None

In [91]:
home_dists = []
away_dists = []

last_games = {} # Mapping of team id to (lat, lon, season) tuple of previous game - if empty, just use np.nan

for i, game in tqdm(all_games.iterrows()):
    
    cur_lat = game['Latitude']
    cur_lon = game['Longitude']
    
    home_team = game['hometeam']
    away_team = game['visteam']
    
    season = game['season']
    
    for team, dists in zip([home_team, away_team], [home_dists, away_dists]):
        if team in last_games:
            last_lat, last_lon, last_season = last_games[team]
            
            if season != last_season: # If last game was a season ago - not always traveling from the game from the past season
                last_lat, last_lon = get_closest_home_game_coords(team, i)
                       
            
        else: # Never played a game before
            last_lat, last_lon = get_closest_home_game_coords(team, i)
            
        if last_lat is not None:
            dist = haversine(last_lat, last_lon, cur_lat, cur_lon)
            dists.append(dist)
        else:
            dists.append(np.nan)
            
        last_games[team] = (cur_lat, cur_lon, season)

all_games['homedistancetraveled'] = home_dists    
all_games['visdistancetraveled'] = away_dists

222340it [01:34, 2363.99it/s] 


## 7. Add Rest Day Columns

In [92]:
home_rest_days = []
away_rest_days = []

last_played = {}

for _, game in tqdm(all_games.iterrows()):
    home_team = game['hometeam']
    away_team = game['visteam']
    timestamp = game['timestamp']
    
    prev_home_t = last_played.get(home_team)
    prev_away_t = last_played.get(away_team)
    
    home_rest_days.append((timestamp.floor('D') - prev_home_t.floor('D')).days if prev_home_t is not None else np.nan)
    away_rest_days.append((timestamp.floor('D') - prev_away_t.floor('D')).days if prev_away_t is not None else np.nan)

    last_played[home_team] = timestamp
    last_played[away_team] = timestamp
 
all_games['homerestdays'] = home_rest_days
all_games['visrestdays'] = away_rest_days

222340it [00:20, 10778.24it/s]


## 8. Add 'homepitcherrgs', 'vispitcherrgs', 'hometeamrgs', 'visteamrgs', 'homepitcherminusteamrgs', and 'vispitcherminusteamrgs' Columns

In [93]:
# Load pitching stats

pitcher_cols = ['gid', 'id', 'team', 'p_gs', 'p_k', 'p_ipouts', 'p_w', 'p_iw', 'p_h', 'p_r', 'p_hr']
pitcher_stats = pd.read_csv('pitching.csv', usecols = pitcher_cols)

# Only starting pitchers
pitcher_stats = pitcher_stats[pitcher_stats['p_gs'] == 1].reset_index(drop=True)

# Add intentional walks to walks
pitcher_stats['p_w'] += pitcher_stats['p_iw']

pitcher_stats = pitcher_stats.drop(['p_gs', 'p_iw'], axis=1)

# If multiple team starters for one game keep first
pitcher_stats = pitcher_stats.drop_duplicates(subset=['gid', 'team'])

pitcher_stats.head()

,gid,id,team,p_ipouts,p_h,p_hr,p_r,p_w,p_k
0,CIN189804150,younc102,CL4,27.0,7.0,0.0,3.0,1.0,2.0
1,CIN189804150,breit101,CIN,27.0,5.0,0.0,2.0,4.0,2.0
2,LS3189804150,killf101,PIT,27.0,13.0,0.0,10.0,3.0,1.0
3,LS3189804150,cunnb103,LS3,27.0,15.0,1.0,3.0,1.0,0.0
4,SLN189804150,grifc101,CHN,27.0,4.0,0.0,1.0,1.0,6.0


In [94]:

(47.4 + pitcher_stats['p_k'] + pitcher_stats['p_ipouts']*1.5 - pitcher_stats['p_w']*2 - pitcher_stats['p_h']*2 - pitcher_stats['p_r']*3 - pitcher_stats['p_hr']*4).mean()

50.5008027076765

In [95]:
# Join with all_games

# On home team
all_games = pd.merge(all_games, pitcher_stats, how='left', left_on=['gid', 'hometeam'], right_on=['gid', 'team'])
all_games = all_games.drop('team', axis=1)
all_games = all_games.rename(columns={'id':'homepitcherid', 'p_k':'homepitcherkouts', 'p_ipouts':'homepitcherouts', 'p_w':'homepitcherwalks', 'p_h':'homepitcherhits', 'p_r':'homepitcherruns', 'p_hr':'homepitcherhomers'})

# On away team
all_games = pd.merge(all_games, pitcher_stats, how='left', left_on=['gid', 'visteam'], right_on=['gid', 'team'])
all_games = all_games.drop('team', axis=1)
all_games = all_games.rename(columns={'id':'vispitcherid',  'p_k':'vispitcherkouts', 'p_ipouts':'vispitcherouts', 'p_w':'vispitcherwalks', 'p_h':'vispitcherhits', 'p_r':'vispitcherruns', 'p_hr':'vispitcherhomers'})

all_games.head()

,gid,visteam,hometeam,site,date,number,starttime,daynight,innings,tiebreaker,usedh,htbf,timeofgame,attendance,fieldcond,precip,sky,temp,winddir,windspeed,oscorer,forfeit,suspend,umphome,ump1b,ump2b,ump3b,umplf,umprf,wp,lp,save,gametype,vruns,hruns,wteam,lteam,line,batteries,lineups,box,pbp,season,homewon,hometeamgamecount,visteamgamecount,marginofvictory,timestamp,Latitude,Longitude,homedistancetraveled,visdistancetraveled,homerestdays,visrestdays,homepitcherid,homepitcherouts,homepitcherhits,homepitcherhomers,homepitcherruns,homepitcherwalks,homepitcherkouts,vispitcherid,vispitcherouts,vispitcherhits,vispitcherhomers,vispitcherruns,vispitcherwalks,vispitcherkouts
0,CIN189804150,CL4,CIN,CIN05,18980415,0.0,0:00PM,day,NaN,NaN,False,True,100.0,11000.0,unknown,unknown,unknown,0.0,unknown,-1.0,NaN,NaN,NaN,sware101,woodg104,NaN,NaN,NaN,NaN,breit101,younc102,NaN,regular,2,3,CIN,CL4,y,both,y,y,NaN,1898,True,1,1,1,1898-04-15 12:00:00,39.116804,-84.535870,0.0,224.919105,NaN,NaN,breit101,27.0,5.0,0.0,2.0,4.0,2.0,younc102,27.0,7.0,0.0,3.0,1.0,2.0
1,LS3189804150,PIT,LS3,LOU03,18980415,0.0,0:00PM,day,NaN,NaN,False,True,80.0,10000.0,unknown,unknown,unknown,0.0,unknown,-1.0,NaN,NaN,NaN,cushc801,heydj901,NaN,NaN,NaN,NaN,cunnb103,killf101,NaN,regular,3,10,LS3,PIT,y,both,y,y,NaN,1898,True,1,1,7,1898-04-15 12:00:00,38.247254,-85.799672,0.0,344.510438,NaN,NaN,cunnb103,27.0,15.0,1.0,3.0,1.0,0.0,killf101,27.0,13.0,0.0,10.0,3.0,1.0
2,SLN189804150,CHN,SLN,STL05,18980415,0.0,0:00PM,day,NaN,NaN,False,NaN,105.0,12000.0,unknown,unknown,unknown,0.0,unknown,-1.0,NaN,NaN,NaN,mcdoj105,odayh101,NaN,NaN,NaN,NaN,grifc101,taylj102,NaN,regular,2,1,CHN,SLN,y,both,y,y,NaN,1898,False,1,1,1,1898-04-15 12:00:00,38.662799,-90.222147,0.0,259.163597,NaN,NaN,taylj102,27.0,7.0,0.0,2.0,1.0,1.0,grifc101,27.0,4.0,0.0,1.0,1.0,6.0
3,BLN189804160,WSN,BLN,BAL07,18980416,0.0,0:00PM,day,NaN,NaN,False,NaN,135.0,6518.0,unknown,unknown,unknown,0.0,unknown,-1.0,NaN,NaN,NaN,lynct901,connt901,NaN,NaN,NaN,NaN,mcjad101,weyhg101,NaN,regular,3,8,BLN,WSN,y,both,y,y,NaN,1898,True,1,1,5,1898-04-16 12:00:00,39.317361,-76.612034,0.0,35.238444,NaN,NaN,mcjad101,27.0,7.0,0.0,3.0,3.0,9.0,weyhg101,24.0,17.0,0.0,8.0,1.0,4.0
4,CIN189804160,CL4,CIN,CIN05,18980416,0.0,0:00PM,day,NaN,NaN,False,True,110.0,6504.0,unknown,unknown,unknown,0.0,unknown,-1.0,NaN,NaN,NaN,woodg104,sware101,NaN,NaN,NaN,NaN,powej105,hillb102,NaN,regular,3,1,CL4,CIN,y,both,y,y,NaN,1898,False,2,2,2,1898-04-16 12:00:00,39.116804,-84.535870,0.0,0.000000,1.0,1.0,hillb102,24.0,7.0,0.0,3.0,3.0,2.0,powej105,27.0,6.0,0.0,1.0,3.0,1.0


In [96]:
# Map from each pitcher id to current rgs and season
pitcher_rgs_map = {}

# Counts for running avgs and season
pitcher_rgs_counts = {}

# Map from each team id to current rgs and season
team_rgs_map = {}

# Counts for running avgs and season
team_rgs_counts = {}

home_individual_rgs = []
away_individual_rgs = []

home_team_rgs = []
away_team_rgs = []

home_first_pitcher_game = []
away_first_pitcher_game = []

INITIAL_RATING = 39 # ~35th percentile of ratings distribution

MEAN_RATING = 49

MEAN_GAME_RATING = 50.5

for _, game in tqdm(all_games.iterrows()):
    home_team = game['hometeam']
    away_team = game['visteam']
    season = game['season']
    
    home_pitcher = game['homepitcherid']
    away_pitcher = game['vispitcherid']
    
    home_k = game['homepitcherkouts']
    away_k = game['vispitcherkouts']
    
    home_outs = game['homepitcherouts']
    away_outs = game['vispitcherouts']
    
    home_walks = game['homepitcherwalks']
    away_walks = game['vispitcherwalks']
    
    home_hits = game['homepitcherhits']
    away_hits = game['vispitcherhits']
    
    home_runs = game['homepitcherruns']
    away_runs = game['vispitcherruns']
    
    home_homers = game['homepitcherhomers']
    away_homers = game['vispitcherhomers']
    
    home_stats = [home_k, home_outs, home_walks, home_hits, home_runs, home_homers]
    away_stats = [away_k, away_outs, away_walks, away_hits, away_runs, away_homers]
    
    for pitcher, team, individual_rgs_lst, team_rgs_lst, first_pitcher_game, stats in zip([home_pitcher, away_pitcher], [home_team, away_team], [home_individual_rgs, away_individual_rgs], [home_team_rgs, away_team_rgs], [home_first_pitcher_game, away_first_pitcher_game], [home_stats, away_stats]):
        
        # 1. Get current ratings
        if pitcher not in pitcher_rgs_map: # Never pitched before-> give base rating
            pitcher_rgs = INITIAL_RATING
            pitcher_rgs_count = 0
            first_pitcher_game.append(True)
            
        elif pitcher_rgs_map[pitcher][1] != season: # New season -> revert to mean by 1/3 and reset count
            pitcher_rgs = pitcher_rgs_map[pitcher][0] + (1/3) * (MEAN_RATING - pitcher_rgs_map[pitcher][0]) # any na will propagate
            pitcher_rgs_count = 0
            first_pitcher_game.append(True)
        
        else:
            pitcher_rgs = pitcher_rgs_map[pitcher][0]
            pitcher_rgs_count = pitcher_rgs_counts[pitcher][0]
            first_pitcher_game.append(False)
            
        individual_rgs_lst.append(pitcher_rgs)
            
        if team not in team_rgs_map : # Never pitched before -> give base rating
            team_rgs = INITIAL_RATING
            team_rgs_count = 0
        elif team_rgs_map[team][1] != season:
            team_rgs = team_rgs_map[team][0] + (1/3) * (MEAN_RATING - team_rgs_map[team][0]) # any na will propagate
            team_rgs_count = 0
        
        else:
            team_rgs = team_rgs_map[team][0]
            team_rgs_count = team_rgs_counts[team][0]
            
        team_rgs_lst.append(team_rgs)
        
        # 2. Update ratings with game results - running averages
        
        # Ensure never nas - assume average gs if no stats available
        na_stats = any(pd.isna(stat) for stat in stats)
        
        # use mean gs if any stats missing
        pitcher_gs = MEAN_GAME_RATING if na_stats else 47.4 + stats[0] + (stats[1])*1.5 - (stats[2])*2 - (stats[3])*2 - (stats[4])*3 - (stats[5])*4
        
        pitcher_eta = 1 / (1 + pitcher_rgs_count)
        
        new_pitcher_rgs = pitcher_eta*pitcher_gs + (1 - pitcher_eta)*pitcher_rgs
        
        pitcher_rgs_map[pitcher] = (new_pitcher_rgs, season)
        pitcher_rgs_counts[pitcher] = (pitcher_rgs_count + 1, season)
        
        team_eta = 1 / (1 + team_rgs_count)
        
        new_team_rgs = team_eta*pitcher_gs + (1 - team_eta)*team_rgs
        
        team_rgs_map[team] = (new_team_rgs, season)
        team_rgs_counts[team] = (team_rgs_count + 1, season)
    
all_games['homepitcherrgs'] = home_individual_rgs
all_games['vispitcherrgs'] = away_individual_rgs
all_games['hometeamrgs'] = home_team_rgs
all_games['visteamrgs'] = away_team_rgs

all_games['homepitcherminusteamrgs'] = all_games['homepitcherrgs']  - all_games['hometeamrgs']
all_games['vispitcherminusteamrgs'] = all_games['vispitcherrgs']  - all_games['visteamrgs']

all_games['homefirstpitchergameofseason'] = home_first_pitcher_game
all_games['visfirstpitchergameofseason'] = away_first_pitcher_game

222340it [00:07, 31304.94it/s]


In [97]:
# Drop temporary pitcher cols
tmp_cols = ['homepitcherid', 'homepitcherkouts', 'homepitcherouts', 'homepitcherwalks', 'homepitcherhits', 'homepitcherruns', 'homepitcherhomers',
            'vispitcherid', 'vispitcherkouts', 'vispitcherouts', 'vispitcherwalks', 'vispitcherhits', 'vispitcherruns', 'vispitcherhomers']

all_games = all_games.drop(tmp_cols, axis=1)

## 9. Add `homewinstreak` and `viswinstreak` Columns

In [98]:
result_map = {}

season_map = {}

home_pcts = []
away_pcts = []

k = 10

for _, row in tqdm(all_games.iterrows()):
    home = row['hometeam']
    away = row['visteam']
    home_won = int(row['homewon'])
    away_won = 1 - home_won
    season = row['season']
    
    for team, pct_lst, won in zip([home, away], [home_pcts, away_pcts], [home_won, away_won]):
        # Check if played before:
        if team not in season_map or season_map[team] != season:
            # Initialize with new list, either if never played or new season
            result_map[team] = deque()
            # Use na % as default
            win_pct = np.nan
            
        else: # They have played a game before this season -> deque is non-empty
            win_pct = sum(result_map[team]) / len(result_map[team])
        
        pct_lst.append(win_pct)
            
        # Update with current result
        if len(result_map[team]) == k:
            result_map[team].popleft()
        
        result_map[team].append(won)
    
        # Update season
        season_map[team] = season
        
all_games['homelastkwinpct'] = home_pcts
all_games['vislastkwinpct'] = away_pcts
    

222340it [00:03, 56718.90it/s]


## 10. Save to `.csv`

In [ ]:
all_games.to_csv(Path.cwd() / "analysis" / "data" / "clean" / "gameinfo_cleaned.csv", date_format='%Y-%m-%d %H:%M:%S', index=False)